# Step 3 — Make new house columns

**Feature engineering** means making a new column from columns we already have. Example: if a house was built in 2000 and sold in 2010, its age at sale was **10 years**.

We will make **four** simple columns. After each one, we check that the result makes sense. Run the cells from top to bottom.

## Load the same sales
We use the first 1,460 rows, as in Steps 1 and 2. `.copy()` lets us add columns without changing the original CSV.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/House_Prices.csv')
sales = df[df['Id'] <= 1460].copy()
print('Houses:', len(sales))

Houses: 1460


## 1. Total area
Add the living area above ground (`GrLivArea`) and the basement area (`TotalBsmtSF`). This gives a rough measure of the space in the house, including the basement. A larger space may mean a higher price.

In [2]:
sales['TotalArea'] = sales['GrLivArea'] + sales['TotalBsmtSF']
print('Total area smaller than living area:',
      (sales['TotalArea'] < sales['GrLivArea']).sum())
print('Largest total area:', sales['TotalArea'].max())

Total area smaller than living area: 0
Largest total area: 11752.0


**Check:** No total is smaller than the living area, so the addition makes sense. One house has a very large total area (**11,752**); we keep it for now and will inspect its prediction error later.

## 2. Total bathrooms
Add full bathrooms and half bathrooms. A half bathroom counts as **0.5**. More bathrooms may make a house more valuable.

In [3]:
sales['TotalBathrooms'] = (sales['FullBath'] + sales['BsmtFullBath']
                           + 0.5 * sales['HalfBath']
                           + 0.5 * sales['BsmtHalfBath'])
print('Fewest bathrooms:', sales['TotalBathrooms'].min())
print('Most bathrooms:', sales['TotalBathrooms'].max())

Fewest bathrooms: 1.0
Most bathrooms: 6.0


**Check:** The result is between **1 and 6** bathrooms, so there are no negative or impossible bathroom counts in these rows.

## 3. House age when sold
Subtract the construction year from the sale year. Age may matter because newer houses often sell for more.

In [4]:
sales['HouseAge'] = sales['YrSold'] - sales['YearBuilt']
print('Houses with negative age:', (sales['HouseAge'] < 0).sum())
print('Oldest house age:', sales['HouseAge'].max())

Houses with negative age: 0
Oldest house age: 136


**Check:** No house has a negative age. The oldest was **136 years old** when sold.

## 4. Years since renovation when sold
Subtract the last renovation year from the sale year. A recent renovation might affect price. If a house was never renovated, `YearRemodAdd` can be its construction year.

In [5]:
sales['YearsSinceRemodel'] = sales['YrSold'] - sales['YearRemodAdd']

bad_rows = sales[sales['YearsSinceRemodel'] < 0]
print('Rows with impossible negative years:', len(bad_rows))
display(bad_rows[['Id', 'YrSold', 'YearRemodAdd']])

Rows with impossible negative years: 1


,Id,YrSold,YearRemodAdd
523,524,2007,2008


**Check:** One row (`Id` **524**) says the renovation happened **one year after** the sale. That cannot describe the house at sale time. For this new feature, we treat it as **0 years** (a very recent renovation). This is a small assumption; the original CSV stays unchanged.

In [6]:
sales['YearsSinceRemodel'] = sales['YearsSinceRemodel'].clip(lower=0)
print('Negative values left:', (sales['YearsSinceRemodel'] < 0).sum())

Negative values left: 0


## Final check
Show the new columns and check for empty values. This lets us see exactly what the next modelling step will receive.

In [7]:
new_columns = ['TotalArea', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel']
display(sales[['Id'] + new_columns].head())
print('Missing values in new columns:')
print(sales[new_columns].isna().sum())

,Id,TotalArea,TotalBathrooms,HouseAge,YearsSinceRemodel
0,1,2566.0,3.5,5,5
1,2,2524.0,2.5,31,31
2,3,2706.0,3.5,7,6
3,4,2473.0,2.0,91,36
4,5,3343.0,3.5,8,8


Missing values in new columns:
TotalArea            0
TotalBathrooms       0
HouseAge             0
YearsSinceRemodel    0
dtype: int64


## Save the result for the next step
This writes a **new** CSV with the four added columns. The original file in `data/raw/` is untouched. The new columns use each house's own information, so they do not learn from other houses.

In [8]:
sales.to_csv('../data/processed/House_Prices_features.csv', index=False)
print('Saved data/processed/House_Prices_features.csv')

Saved data/processed/House_Prices_features.csv


## What we made

| New column | Simple meaning | Why it may help |
|---|---|---|
| `TotalArea` | Living area + basement | Describes more of the house's space |
| `TotalBathrooms` | Full baths + half baths counted as 0.5 | Describes bathroom convenience |
| `HouseAge` | Sale year − construction year | Describes how old the house was |
| `YearsSinceRemodel` | Sale year − renovation year | Describes how recent the renovation was |

All four columns have values for the 1,460 houses. The single inconsistent renovation row was shown and handled. **Next:** train and compare models using this prepared source data.